In [1]:
using Pkg
Pkg.resolve()

# Pkg.add(PackageSpec(name="InformationMeasures", version = "0.3.0"))
# Pkg.add(PackageSpec(name="InformationMeasures", version = "0.3.0"))

# Pkg.add(PackageSpec(name="PyPlot", version = "2.8.0"))

# Pkg.add(PackageSpec(name="LightGraphs", version="1.2.0"))

# # Pkg.add(PackageSpec(name = "GraphPlot", version = "0.3.1"))
# Pkg.add(PackageSpec(name="CSV"))
# Pkg.add("DataFrames")

# Hopefully this step precompiles the libraries
using InformationMeasures
using LightGraphs
using CSV, DataFrames, InformationMeasures

  No Changes to `/projects/b1042/GoyalLab/Keerthana/grnInference/code/InformationMeasures.jl/keerthana_test/Project.toml`
  No Changes to `/projects/b1042/GoyalLab/Keerthana/grnInference/code/InformationMeasures.jl/keerthana_test/Manifest.toml`


In [2]:
# Functions that implement formulae for the information measures

# Information measures are reviewed in:
# Timme, Nicholas; Alford, Wesley; Flecker, Benjamin; Beggs, John M. (2013-07-03).
# "Synergy, redundancy, and multivariate information measures: an experimentalist's perspective".
# Journal of Computational Neuroscience. 36 (2): 119–140.
# http://link.springer.com/article/10.1007%2Fs10827-013-0458-4

# TODO: Add further measures (dual total correlation, delta i)

export apply_entropy_formula, apply_conditional_entropy_formula, apply_mutual_information_formula,
    apply_conditional_mutual_information_formula, apply_interaction_information_formula,
    apply_total_correlation_formula, apply_specific_information_formula, apply_redundancy_formula,
    apply_unique_information_formula, apply_synergy_formula, apply_cross_entropy_formula

function remove_non_finite(x)
    return isfinite(x) ? x : zero(x)
end

function remove_negative(x)
    return x < 0 ? zero(x) : x
end

# Parameters:
# 	- probabilities, array of floats
# 	- base, number
function apply_entropy_formula(p::AbstractArray{T}, base::R) where {T<:AbstractFloat,R<:Real}
    return -sum(remove_non_finite.(p .* log.(base, p)))
end

# Parameters:
# 	- joint entropy of all variables, number
#	- entropy of conditioned variables, number
function apply_conditional_entropy_formula(entropy_xy, entropy_y)
    return entropy_xy - entropy_y
end

# Parameters:
# 	- joint probabilities, array of floats
# 	- probabilities (first variable), array of floats
# 	- probabilities (second variable), array of floats
# 	- base, number
function apply_mutual_information_formula(p_xy::AbstractArray{T}, p_x::AbstractArray{T}, p_y::AbstractArray{T}, base::R) where {T<:AbstractFloat,R<:Real}
    return sum(remove_non_finite.(p_xy .* log.(base, p_xy ./ (p_x .* p_y))))
end
# Parameters:
# 	- entropy of first variable, number
# 	- entropy of second variable, number
#	- joint entropy of both variables, number
function apply_mutual_information_formula(entropy_x, entropy_y, entropy_xy)
    return entropy_x + entropy_y - entropy_xy
end

# Parameters:
# 	- joint entropy of first and conditioned variables, number
# 	- joint entropy of second and conditioned variables, number
#	- joint entropy of all three variables, number
function apply_conditional_mutual_information_formula(entropy_xz, entropy_yz, entropy_xyz, entropy_z)
    return entropy_xz + entropy_yz - entropy_xyz - entropy_z
end

# Parameters:
# 	- mutual information of first two variables, number
# 	- conditional mutual information of first two variables on the third, number
function apply_interaction_information_formula(conditional_mutual_information, mutual_information)
    return conditional_mutual_information - mutual_information
end

# Parameters:
# 	- entropy of first variable, number
# 	- entropy of second variable, number
# 	- entropy of third variable, number
#	- joint entropy of all three variables, number
function apply_total_correlation_formula(entropy_x, entropy_y, entropy_z, entropy_xyz)
    return entropy_x + entropy_y + entropy_z - entropy_xyz
end

# Parameters:
# 	- joint probabilities, array of floats
# 	- probabilities (source), array of floats
# 	- probabilities (target), array of floats
# 	- dimension along which to sum, integer
# 	- base, number
function apply_specific_information_formula(p_xz, p_x, p_z, dim_sum, base)
    return vec(sum(remove_non_finite.((p_xz ./ p_z) .* log.(base, p_xz ./ (p_x .* p_z))), dims=dim_sum))
end

# Parameters:
# 	- joint probabilities (source 1 and target), array of floats
# 	- joint probabilities (source 2 and target), array of floats
# 	- probabilities (source 1), array of floats
# 	- probabilities (source 2), array of floats
# 	- probabilities (target), array of floats
# 	- dimensions along which to sum, tuple of integers
# 	- base, number
function apply_redundancy_formula(p_xz::AbstractArray{T}, p_yz::AbstractArray{T}, p_x::AbstractArray{T}, p_y::AbstractArray{T}, p_z::AbstractArray{T}, dim_sum::Tuple{I,I}, base::R) where {T<:AbstractFloat,R<:Real,I<:Integer}
    minimum_specific_information = min.(
        apply_specific_information_formula(p_xz, p_x, p_z, dim_sum[1], base),
        apply_specific_information_formula(p_yz, p_y, p_z, dim_sum[2], base)
    )
    return sum(vec(p_z) .* vec(minimum_specific_information))
end
# Parameters:
# 	- probabilities (target), array of floats
# 	- specific information of source 1 and target, array of floats
# 	- specific information of source 2 and target, array of floats
# 	- base, number
function apply_redundancy_formula(p_z::AbstractArray{T}, specific_information_1::AbstractArray{T}, specific_information_2::AbstractArray{T}, base::R) where {T<:AbstractFloat,R<:Real}
    minimum_specific_information = min.(specific_information_1, specific_information_2)
    return sum(vec(p_z) .* vec(minimum_specific_information))
end

# Parameters:
# 	- mutual information of source 1 and target, number
# 	- redundancy of both sources and target, number
function apply_unique_information_formula(mutual_information, redundancy)
    # Rounding errors may lead to slightly negative results
    return remove_negative.(mutual_information - redundancy)
end

# Parameters:
# 	- interaction information of both sources and target, number
# 	- redundancy of both sources and target, number
function apply_synergy_formula(interaction_information, redundancy)
    return interaction_information + redundancy
end

# Parameters:
#	- probabilities of first variable
#	- probabilities of second variable
#	- base of exponent
function apply_cross_entropy_formula(p_x::AbstractArray{T}, p_y::AbstractArray{T}, base::R) where {T<:AbstractFloat,R<:Real}
    return -sum(remove_non_finite.(p_x .* log.(base, p_y)))
end


apply_cross_entropy_formula (generic function with 1 method)

In [3]:
# Gets the joint probability distribution for two Nodes.
function get_joint_probabilities(node1, node2, estimator)

    frequencies = get_frequencies_from_bin_ids(
        node1.binned_values,
        node2.binned_values,
        node1.number_of_bins,
        node2.number_of_bins
    )

    probabilities = get_probabilities(estimator, frequencies)
    # probabilities is already property of a node, but doing this gets correct array shapes.
    # Also, for MI and CLR, it means that we don't assume that the marginal probabilities for
    # a node are always the same, no matter what the second node is, meaning that we can use
    # estimators other than maximum likelihood. (We still can't do this for PUC and PIDC,
    # because we do make that assumption for 3-node joint distributions, in get_puc.)
    probabilities1 = sum(probabilities, dims=2)
    probabilities2 = sum(probabilities, dims=1)

    return (probabilities, probabilities1, probabilities2)

end

get_joint_probabilities (generic function with 1 method)

In [13]:
function get_puc_scores(nodes, number_of_nodes, base)

    function get_mi_and_si(node1, node2, base) # Mutual information and specific information
        probabilities, probabilities1, probabilities2 = get_joint_probabilities(node1, node2, estimator)
        mi = apply_mutual_information_formula(probabilities, probabilities1, probabilities2, base)
        si1 = apply_specific_information_formula(probabilities, probabilities1, probabilities2, 1, base)
        si2 = apply_specific_information_formula(probabilities, probabilities2, probabilities1, 2, base)
        return (mi, si1, si2)
    end

    function get_node_pairs(node1, node2, i, j, base)
        mi, si1, si2 = get_mi_and_si(node1, node2, base)
        node_pairs[i, j] = NodePair(mi, si1)
        node_pairs[j, i] = NodePair(mi, si2)
    end

    function increment_puc_scores(x, z, mi, redundancy, puc_scores)
        puc_score = (mi - redundancy) / mi
        puc_score = isfinite(puc_score) && puc_score >= 0 ? puc_score : zero(puc_score)
        puc_scores[x, z] += puc_score
        puc_scores[z, x] += puc_score
    end

    function get_puc(target, source1_target, source2_target, x, y, z, puc_scores)
        redundancy = apply_redundancy_formula(
            target.probabilities,
            source1_target.si,
            source2_target.si,
            base
        )
        println("Redundancy is $(redundancy)")
        println("source1_target MI is $(source1_target.mi)")
        println("source2_target MI is $(source2_target.mi)")
        increment_puc_scores(x, z, source1_target.mi, redundancy, puc_scores)
        increment_puc_scores(y, z, source2_target.mi, redundancy, puc_scores)
    end

    node_pairs = Array{NodePair}(undef, number_of_nodes, number_of_nodes)
    puc_scores = SharedArray{Float64}(number_of_nodes, number_of_nodes)

    for i in 1:number_of_nodes
        for j in i+1:number_of_nodes
            get_node_pairs(nodes[i], nodes[j], i, j, base)
        end
    end

    for i in 1:number_of_nodes
        for j in i+1:number_of_nodes
            for k in j+1:number_of_nodes
                get_puc(nodes[k], node_pairs[i, k], node_pairs[j, k], i, j, k, puc_scores)
                get_puc(nodes[j], node_pairs[i, j], node_pairs[k, j], i, k, j, puc_scores)
                get_puc(nodes[i], node_pairs[j, i], node_pairs[k, i], j, k, i, puc_scores)
            end
        end
    end

    return puc_scores

end

get_puc_scores (generic function with 1 method)

In [5]:
struct Node
    label::String
    binned_values::Array{Int64}
    number_of_bins::Int64
    probabilities::Array{Float64}
end

# Constructs a Node from a line of a data file. line should be an array with
# the label as the first element, then the raw data values.
function Node(line::AbstractArray, discretizer, estimator, number_of_bins)

    label = string(line[1])
    raw_values = Array{Float64}(line[2:end])

    # Raw values are mapped to their bin IDs
    binned_values = zeros(Int, length(raw_values))

    # If the discretizer is Bayesian blocks, number_of_bins will be
    # overwritten by the ideal number of bins. Otherwise, it will remain
    # the same as the value passed in.
    number_of_bins = get_bin_ids!(raw_values, discretizer, number_of_bins, binned_values)

    probabilities = get_probabilities(estimator, get_frequencies_from_bin_ids(binned_values, number_of_bins))

    return Node(label, binned_values, number_of_bins, probabilities)

end

# Type for caching information between pairs of nodes:
# - mi: mutual information
# - si: specific information
struct NodePair
    mi::Float64
    si::Array{Float64}
end

"""
Undirected edge

Fields:
* `nodes`: the two nodes, in an arbitrary order
* `weight`: weight indicating confidence of edge existing in the true network
Weights are used to rank the edges, and different algorithms may have a
different scale. The relative weights within one inferred network are
therefore more meaningful than the absolute weight out of context.
"""
struct Edge
    nodes::Array{Node}
    weight::Float64
end

Edge

In [20]:
using CSV, DataFrames, SharedArrays

# --- Load your data ---
df = CSV.read("/home/mzo5929/Keerthana/grnInference/code/InformationMeasures.jl/keerthana_test/x_combined_fan_out_4.csv", DataFrame)

# --- Choose discretization and estimator options ---
discretizer = "equal_width"        # or "equal_width"
estimator = "maximum_likelihood"       # must be String, not Symbol
number_of_bins = 10                    # You can tune this as needed

# --- Construct Node objects using their constructor ---
nodes = [
    Node([col; df[!, col]], discretizer, estimator, number_of_bins)
    for col in names(df)
]
# Define parameters
number_of_nodes = length(nodes)
base = 2.0

# Call get_puc_scores
puc_scores = get_puc_scores(nodes, number_of_nodes, base)

Mode doesn't exist, fell back to uniform width
Mode doesn't exist, fell back to uniform width
Mode doesn't exist, fell back to uniform width
Redundancy is 0.00397812628129208
source1_target MI is 0.1484016131904335
source2_target MI is 0.00397812628129208
Redundancy is 0.00397812628129208
source1_target MI is 0.14210847691857967
source2_target MI is 0.00397812628129208
Redundancy is 0.13739819699283123
source1_target MI is 0.14210847691857967
source2_target MI is 0.1484016131904335


3×3 SharedMatrix{Float64}:
 0.0      1.00515  1.04734
 1.00515  0.0      0.0
 1.04734  0.0      0.0